# Confidence-Based Tool Routing — run on Colab

Runs the whole experiment on a free Colab GPU: nothing written to your own
machine, no API key, no quota. A T4 serves an 8B model comfortably, which is
larger than an 8GB laptop can hold.

**Set a GPU runtime first:** Runtime → Change runtime type → T4 GPU.
It works on CPU too, just far slower.

The model is served by `ollama` because its OpenAI-compatible endpoint
returns `logprobs` and `top_logprobs`. That was verified against a live
server rather than taken from docs — its own compatibility page omits them.
Those logprobs are what the token-entropy estimator reads, and entropy is
the only one of the four estimators that costs nothing, so losing it would
cost the cost argument.

## 1. Install and start the server

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import os, subprocess, time, requests

# Models go to /content so nothing touches your own disk, and Colab has
# no service manager, so the server runs as a background process.
os.makedirs('/content/ollama', exist_ok=True)
os.environ['OLLAMA_MODELS'] = '/content/ollama'
subprocess.Popen(['ollama', 'serve'],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(60):
    try:
        requests.get('http://localhost:11434/api/version', timeout=2)
        print('server up'); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError('ollama did not start')

## 2. Pull the model

`llama3.1:8b` fits a T4 and is the right *strength* for this experiment, not
just the right size. A frontier model solves ~95% of GSM8K unaided, so almost
nothing gets labelled `tool_necessity=required` and routing recall is computed
over an almost empty positive class. A weaker model fails often enough to
leave something to measure.

It is also served commercially, which matters for step 5.

In [ ]:
MODEL = 'llama3.1:8b'      # 'llama3.2:3b' if the GPU is contended
!ollama pull {MODEL}

## 3. Get the harness

In [ ]:
!git clone -q https://github.com/confidence-routing/confidence-tool-routing.git /content/repo
%cd /content/repo/confidence_routing
!git checkout -q integration/phase4
!pip install -q openai
!python -m pytest tests/ -q 2>&1 | tail -2

## 4. Confirm the model returns logprobs

One cheap call. Without logprobs the entropy estimator scores `None` on every
task — worth discovering here rather than at the end of a finished sweep.

In [ ]:
!python -m examples.run_experiment --probe --provider ollama --model {MODEL}

## 5. The rate to price it at

The run is free, so CPST reports what it **would** have cost at this model's
paid rate. That is the only way to compare estimators whose entire difference
is price, and the rule that keeps it defensible is that a model is priced at
its *own* rate on a *named* provider — never a substitute's.

So the rate is passed on the command line rather than edited into the source:
the provenance travels with the command that produced the numbers.

**Look these up before trusting any CPST figure.** The placeholder below is
rejected on purpose — `run_experiment` exits rather than computing a cost from
a rate nobody can trace, because CPST refusing to compute is visible and a
made-up number is not.

In [ ]:
PRICE_IN  = 0.18                            # USD per 1M input tokens
PRICE_OUT = 0.18                            # USD per 1M output tokens
PRICE_SRC = 'REPLACE: provider + date'      # e.g. 'together.ai 2026-09-25'

PRICING = f'--price-in {PRICE_IN} --price-out {PRICE_OUT} \\
           --price-source "{PRICE_SRC}"'
print(PRICING)

## 6. Label `tool_necessity`

Attempts each task with no tool, three times. Always right means it never
needed one; never right means it did; sometimes is `ambiguous` and excluded
from routing metrics by default.

Only half the tasks are labelled — the rest are held out, because scoring the
router on the same tasks whose labels came from the model's own failures makes
*needed a tool* and *the router escalated* two readings of one measurement.

**Read the summary.** If nearly everything is `not_required`, this dataset is
ceilinged for this model and routing cannot be measured on it whatever the
estimator does. It warns below 10%.

In [ ]:
!python -m examples.label_pilot --dataset gsm8k --limit 200 \\
    --provider ollama --model {MODEL} --trials 3

## 7. Run the four estimators

Same tasks, same threshold, four different confidence signals. The comparison
between them *is* the experiment — entropy is free, self-consistency costs k
extra samples, the verifier costs one extra call.

In [ ]:
for method in ['entropy', 'self_consistency', 'external_llm', 'hybrid']:
    print('=' * 72, f'\n{method}\n', '=' * 72)
    !python -m examples.run_experiment --dataset gsm8k --limit 100 \\
        --provider ollama --model {MODEL} --verifier-model {MODEL} \\
        --method {method} --threshold 0.7 {PRICING}

## 8. Save the logs

Colab disks are ephemeral and the session ends when idle. One JSONL row per
task means the logs *are* the results, so this step is not optional.

In [ ]:
!zip -qr /content/runs.zip runs labels
from google.colab import files
files.download('/content/runs.zip')

## Notes

- `OLLAMA_MODELS` points at `/content`, so nothing is written to your machine.
- Repeat steps 6–7 for `coqa` and `humaneval`. Two tool categories is the
  minimum for the cross-tool transfer comparison, which is the headline result.
- To find a threshold rather than assume 0.7, `sweep_thresholds()` replays a
  finished log at every cut and needs no extra calls.
- The verifier here is the same model as the generator, which is the cheap
  option but not the interesting one: a verifier's value is having different
  blind spots from the model it checks. Worth a second model once the pipeline
  is producing numbers.